# 3T LCModel metabolite-map comparison

Publication-style comparison with a high-resolution anatomical localizer, two axial MRSI slices, and shared color scaling per metabolite.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path('.').resolve()

SUBJECT = Path('PRISMA_Vienna/Vol05_LH/Res50x50')
SUPERIOR_SLICE = 15  # A, MRSI z-index
INFERIOR_SLICE = 6  # B, MRSI z-index

PIPELINES = [
    ('No suppression', 'LCModelB4Walinet'),
    ('HLSVD + L2',             'LCModelHSVDL2'),
    ('WALRUS',         'LCModelAfterWalinet'),
]
METABOLITES = {
    'tNAA': 'NAA_NAAG',
    'tCr':  'Cr_PCr',
    'tCho': 'GPC_PCh',
    'Glx':  'Glu_Gln',
}

VMAX_PERCENTILE = 98  # gemeinsame Skala über beide Slices und alle Methoden
ANATOMY_UPSAMPLING = 4
MAP_ALPHA = 1  # T1 bleibt unter der Metabolitkarte sichtbar
LOCALIZER_ORIENTATION = 'sagittal'  # 'coronal' oder 'sagittal'
COLORMAP = 'viridis'
FIGSIZE = (11.8, 9.4)  # kompakte Spaltenabstände wie in der 7T-Figure

SAVE_PDF =  "SavedGraphics/3T_metabolite_maps.pdf"  # z.B. PROJECT_ROOT / 'results/PaperFigures/3T_metabolite_maps.pdf'

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.metabolite_map_figure import create_figure

b0_dir = PROJECT_ROOT / 'data/3T/B0corrected_wo_LipidMask' / SUBJECT
no_b0_dir = PROJECT_ROOT / 'data/3T/NoB0Correction' / SUBJECT

figure, qc = create_figure(
    b0_subject_dir=b0_dir,
    no_b0_subject_dir=no_b0_dir,
    pipelines=PIPELINES,
    metabolites=METABOLITES,
    superior_slice=SUPERIOR_SLICE,
    inferior_slice=INFERIOR_SLICE,
    vmax_percentile=VMAX_PERCENTILE,
    anatomy_upsampling=ANATOMY_UPSAMPLING,
    cmap=COLORMAP,
    map_alpha=MAP_ALPHA,
    localizer_orientation=LOCALIZER_ORIENTATION,
    figsize=FIGSIZE,
)

if SAVE_PDF is not None:
    save_path = Path(SAVE_PDF).expanduser()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(save_path, format='pdf', bbox_inches='tight')
    print(f'Saved PDF: {save_path.resolve()}')

print(f"MRSI grid: {qc['mask_shape']}")
print(f"Original anatomy: {qc['anatomy_shape_original']}")
print(f"MRSI-aligned high-resolution anatomy: {qc['anatomy_shape_resampled']}")
print(f"Map orientation transform: {qc['array_orientation_transform']} "
      f"(mask Dice={qc['array_orientation_dice']:.5f})")